# 07 · Detector de placas a ONNX int8 (para la PWA del guardia)

Se ejecuta en **Google Colab**, de arriba abajo. La app de portería (Erick) corre el detector **en el dispositivo, sin red** (O6), con
`onnxruntime-web`. Para eso el modelo se exporta a ONNX y se **cuantiza a int8** (más chico y más rápido).

**Cómo se evita cuantizar a ciegas.** Se mide mAP@0.5 de PyTorch, ONNX fp32 y varias variantes int8 en las **mismas 65 imágenes de prueba** del
notebook 04 (mismo dataset, misma semilla, misma partición). La calibración del int8 usa solo imágenes de **entrenamiento**. El int8 solo se
recomienda si pierde poco frente a PyTorch (`caida_maxima_ap50` en `ml/configs/placas_onnx.yaml`); si no, se entrega el fp32.

Se usa cuantización **estática en formato QDQ**, la que `onnxruntime-web` ejecuta bien; la dinámica falla con muchas redes convolucionales en WASM.

**Antes de empezar:** sube `ml.zip` y `yolov8n_placas.pt` (de `D:\SW2\modelos\placas\2026-10-15_placas_deteccion\`).

In [ ]:
import os, shutil, zipfile
from google.colab import files

subidos = files.upload()          # ml.zip y yolov8n_placas.pt
os.makedirs("/content/condohub-ia", exist_ok=True)
for n in subidos:                 # Colab renombra a "ml (1).zip" si ya existía
    if n.lower().startswith("ml"):
        zipfile.ZipFile(n).extractall("/content/condohub-ia")
    elif n.lower().endswith(".pt"):
        shutil.move(n, "/content/yolov8n_placas.pt")
os.environ["ML_DATA_DIR"] = "/content/condohub-ml"
os.chdir("/content/condohub-ia")
print("pesos:", os.path.exists("/content/yolov8n_placas.pt"), "| carpeta:", os.listdir("/content/condohub-ia"))

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxslim kagglehub pyyaml
import onnxruntime
print("onnxruntime", onnxruntime.__version__)

## 1 · Exportar a ONNX (fp32)

In [ ]:
import json, time, hashlib, yaml, numpy as np
from datetime import date
from pathlib import Path
from PIL import Image
from ultralytics import YOLO
from ml.src import placas, onnx_placas as op
from ml.src.paths import dir_datos, dir_experimento

cfg = yaml.safe_load(open("ml/configs/placas_onnx.yaml", encoding="utf-8"))
cfg_det = yaml.safe_load(open("ml/configs/placas_deteccion.yaml", encoding="utf-8"))
TAM = cfg["onnx"]["tamano_imagen"]

torch_modelo = YOLO("/content/yolov8n_placas.pt")
ruta_fp32 = torch_modelo.export(format="onnx", imgsz=TAM, opset=cfg["onnx"]["opset"], simplify=True, dynamic=False)
print(ruta_fp32, round(os.path.getsize(ruta_fp32) / 1e6, 1), "MB")

## 2 · Los mismos datos del notebook 04 (misma partición)

In [ ]:
import kagglehub

origen = Path(kagglehub.dataset_download(cfg_det["dataset"]["kaggle"]))
imagenes = {p.stem: p for ext in ("png", "jpg", "jpeg") for p in origen.rglob(f"*.{ext}")}
anotaciones = {p.stem: p for p in origen.rglob("*.xml")}
comunes = sorted(set(imagenes) & set(anotaciones))
partes = placas.dividir_imagenes(comunes, tuple(cfg_det["dataset"]["proporciones"]), cfg_det["semilla"])
print({k: len(v) for k, v in partes.items()})          # debe ser 303 / 65 / 65 como en el notebook 04
assert len(comunes) == 433

## 3 · Cuantizar a int8 (estático, QDQ) con imágenes de calibración de entrenamiento

Se prueban tres variantes y se elige con datos, no a ojo: `u8u8` y `u8s8` (tipo de los pesos) y `u8s8_solo_conv` (solo se cuantizan las convoluciones;
la cabeza de detección suele ser lo más sensible a la cuantización).

In [ ]:
import onnxruntime as ort
from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static

def cargar(ruta):
    return np.asarray(Image.open(ruta).convert("RGB"))

sesion_fp32 = ort.InferenceSession(ruta_fp32, providers=["CPUExecutionProvider"])
entrada = sesion_fp32.get_inputs()[0]
print("entrada:", entrada.name, entrada.shape, entrada.type)
print("salida :", [(o.name, o.shape) for o in sesion_fp32.get_outputs()])

class LectorCalibracion(CalibrationDataReader):
    def __init__(self, rutas):
        self._it = iter(rutas)
    def get_next(self):
        ruta = next(self._it, None)
        return None if ruta is None else {entrada.name: op.letterbox(cargar(ruta), TAM)[0]}

nombres_calibracion = partes["train"][: cfg["onnx"]["calibracion_imagenes"]]
assert not set(nombres_calibracion) & set(partes["test"])          # nunca imágenes de prueba
rutas_calibracion = [imagenes[n] for n in nombres_calibracion]

base = ruta_fp32
try:                                       # paso recomendado antes de cuantizar; si falla se sigue sin él
    from onnxruntime.quantization.shape_inference import quant_pre_process
    quant_pre_process(ruta_fp32, "/content/yolov8n_placas_pre.onnx", skip_symbolic_shape=True)
    base = "/content/yolov8n_placas_pre.onnx"
except Exception as e:
    print("preproceso opcional omitido:", type(e).__name__, e)

variantes = {
    "u8u8": dict(activation_type=QuantType.QUInt8, weight_type=QuantType.QUInt8),
    "u8s8": dict(activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8),
    "u8s8_solo_conv": dict(activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, op_types_to_quantize=["Conv"]),
}
rutas_int8 = {}
for nombre, opciones in variantes.items():
    destino_onnx = f"/content/yolov8n_placas_int8_{nombre}.onnx"
    t0 = time.time()
    quantize_static(base, destino_onnx, LectorCalibracion(rutas_calibracion),
                    quant_format=QuantFormat.QDQ, per_channel=False, **opciones)
    rutas_int8[nombre] = destino_onnx
    print(f"{nombre}: {os.path.getsize(destino_onnx) / 1e6:.1f} MB ({time.time() - t0:.0f} s)")

## 4 · Comparar con PyTorch en las imágenes de prueba

Mismo posproceso para todos los ONNX (`ml/src/onnx_placas.py`, la referencia que replicará la PWA). Se miden: mAP@0.5, precisión y recall en el punto
de operación de la app (`conf_minima`), cuántas detecciones de PyTorch reproduce cada ONNX, latencia en CPU y tamaño.

In [ ]:
CONF_BAJA = 0.01                      # para calcular el AP se conservan también las detecciones débiles
CONF_APP = cfg["onnx"]["conf_minima"]
IOU = cfg["onnx"]["iou_nms"]
rutas_prueba = [imagenes[n] for n in partes["test"]]
verdad = [np.array(placas.voc_a_cajas(anotaciones[n].read_text(encoding="utf-8")), dtype=float).reshape(-1, 4)
          for n in partes["test"]]
print("imágenes de prueba:", len(rutas_prueba), "| placas de referencia:", sum(len(v) for v in verdad))

def en_pytorch(rutas):
    salidas = []
    for r in rutas:
        b = torch_modelo.predict(str(r), conf=CONF_BAJA, iou=IOU, imgsz=TAM, verbose=False)[0].boxes
        salidas.append(np.column_stack([b.xyxy.cpu().numpy(), b.conf.cpu().numpy()]) if len(b) else np.zeros((0, 5)))
    return salidas

def en_onnx(ruta_onnx, rutas):
    s = ort.InferenceSession(ruta_onnx, providers=["CPUExecutionProvider"])
    nombre_entrada = s.get_inputs()[0].name
    salidas, tiempos = [], []
    for r in rutas:
        arr = cargar(r)
        tensor, escala, px, py = op.letterbox(arr, TAM)
        t0 = time.perf_counter()
        y = s.run(None, {nombre_entrada: tensor})[0]
        tiempos.append(time.perf_counter() - t0)
        salidas.append(op.decodificar(y, escala, px, py, arr.shape[:2], conf_min=CONF_BAJA, iou_nms=IOU))
    return salidas, float(np.median(tiempos)) * 1000

pred_torch = en_pytorch(rutas_prueba)
modelos = {"pytorch": (pred_torch, None, os.path.getsize("/content/yolov8n_placas.pt") / 1e6)}
for nombre, ruta in {"onnx_fp32": ruta_fp32, **{f"onnx_int8_{n}": r for n, r in rutas_int8.items()}}.items():
    pred, ms = en_onnx(ruta, rutas_prueba)
    modelos[nombre] = (pred, ms, os.path.getsize(ruta) / 1e6)

filas = {}
for nombre, (pred, ms, mb) in modelos.items():
    filas[nombre] = {
        "ap50": op.ap50(pred, verdad), **op.precision_recall(pred, verdad, CONF_APP),
        "latencia_cpu_ms": ms, "tamano_mb": round(mb, 2),
        "coincidencia_con_pytorch": op.coincidencia(pred_torch, pred, CONF_APP),
    }

def f(x, d=3):
    return "  -  " if x is None else f"{x:.{d}f}"

print(f"{'modelo':<24}{'AP50':>7}{'prec':>7}{'recall':>8}{'MB':>7}{'ms CPU':>8}   coincide con PyTorch (conf >= {CONF_APP})")
for n, r in filas.items():
    c = r["coincidencia_con_pytorch"]
    print(f"{n:<24}{f(r['ap50']):>7}{f(r['precision']):>7}{f(r['recall']):>8}{r['tamano_mb']:>7.1f}{f(r['latencia_cpu_ms'], 0):>8}   "
          f"{c['coinciden']} comunes, {c['solo_el_primero']} solo PyTorch, {c['solo_el_segundo']} solo ONNX")

In [ ]:
ap_ref = filas["pytorch"]["ap50"]
candidatas = {n: r for n, r in filas.items() if n.startswith("onnx_int8")}
mejor = max(candidatas, key=lambda n: (candidatas[n]["ap50"], -candidatas[n]["tamano_mb"]))
caida = ap_ref - candidatas[mejor]["ap50"]
decision = "ADOPTAR_INT8" if caida <= cfg["onnx"]["caida_maxima_ap50"] else "USAR_FP32"
print(f"mejor int8: {mejor} | AP50 {candidatas[mejor]['ap50']:.3f} frente a PyTorch {ap_ref:.3f} | caída {caida:.3f} "
      f"(máximo aceptado {cfg['onnx']['caida_maxima_ap50']}) -> {decision}")

## 5 · Guardar y descargar

Se guardan el modelo elegido y su ficha (`modelo_placas.json`: entrada, salida, posproceso y `sha256`, lo que pide `GET /modelos/placas/actual`).
El `.onnx` **no** va a git (I6): se publica en el almacén S3 y se guarda en `D:\SW2\modelos\placas`.

In [ ]:
from google.colab import files

destino = dir_experimento(cfg["nombre"])
pesos = dir_datos() / "modelos" / "placas" / f"{date.today().isoformat()}_onnx"
pesos.mkdir(parents=True, exist_ok=True)
int8 = decision == "ADOPTAR_INT8"
origen_elegido = rutas_int8[mejor.replace("onnx_int8_", "")] if int8 else ruta_fp32
nombre_archivo = f"yolov8n_placas_{'int8' if int8 else 'fp32'}.onnx"
shutil.copy(origen_elegido, pesos / nombre_archivo)
sesion = ort.InferenceSession(str(pesos / nombre_archivo), providers=["CPUExecutionProvider"])

ficha = {
    "modelo": "yolov8n-placas", "version": cfg["onnx"]["version"],
    "formato": "onnx-int8" if int8 else "onnx-fp32", "archivo": nombre_archivo,
    "sha256": hashlib.sha256((pesos / nombre_archivo).read_bytes()).hexdigest(),
    "tamanoBytes": (pesos / nombre_archivo).stat().st_size, "opset": cfg["onnx"]["opset"],
    "entrada": {"nombre": sesion.get_inputs()[0].name, "forma": sesion.get_inputs()[0].shape, "tipo": sesion.get_inputs()[0].type,
                "preproceso": "RGB, redimensionar sin deformar a 640 (letterbox, relleno gris 114), dividir entre 255, NCHW"},
    "salida": {"nombre": sesion.get_outputs()[0].name, "forma": sesion.get_outputs()[0].shape,
               "significado": "[1, 5, 8400]: por candidata cx, cy, w, h (píxeles del tensor de 640) y confianza"},
    "posproceso": {"confianzaMinima": CONF_APP, "iouNms": IOU, "referencia": "ml/src/onnx_placas.py (decodificar, nms)"},
    "decision": decision,
}
resultado = {"config": cfg, "modelos": filas, "mejor_int8": mejor, "caida_ap50": caida, "decision": decision,
             "particion": {k: len(v) for k, v in partes.items()}, "imagenes_calibracion": len(rutas_calibracion)}
dump = lambda o: o.item() if hasattr(o, "item") else str(o)
(destino / "metrics.json").write_text(json.dumps(resultado, indent=2, ensure_ascii=False, default=dump), encoding="utf-8")
(destino / "modelo_placas.json").write_text(json.dumps(ficha, indent=2, ensure_ascii=False, default=dump), encoding="utf-8")
print(json.dumps(ficha, indent=2, ensure_ascii=False, default=dump))

shutil.copy(ruta_fp32, pesos / "yolov8n_placas_fp32.onnx")     # respaldo: la PWA lo usa si no carga el int8 (nodos QDQ)
for ruta in (pesos / nombre_archivo, pesos / "yolov8n_placas_fp32.onnx", destino / "metrics.json", destino / "modelo_placas.json"):
    files.download(str(ruta))